# Create KEGG reference files

1. REF_KEGG2LABEL: dictionary of KEGG to label
2. REF_KEGG2FORMULA: dictionary of KEGG to shortened formula
3. REF_KEGG2NAMES: dictionary of KEGG to all synonyms

In [ ]:
import urllib.request
import gzip
import compress_pickle
import os
import string
import pandas as pd
import numpy as np
import re


import os
import requests
import time
import lzma
import json
import os
import json
import glob
import re

# Data

Data was obtained on July 22, 2025 from [www.kegg.jp](www.kegg.jp)

In [ ]:
# download reactions from KEGG
import os
import requests
import time

OUTPUT_DIR = "kegg_data/reaction_flats"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Step 1: Get all reaction IDs
def download_all_reaction_ids():
    url = "https://rest.kegg.jp/list/reaction"
    print(f"Fetching all KEGG reaction IDs from {url}...")
    r = requests.get(url)
    if r.status_code != 200:
        raise Exception(f"Failed to fetch reaction list: {r.status_code}")
    
    lines = r.text.strip().split("\n")
    rxn_ids = [line.split()[0].replace("rn:", "") for line in lines]
    with open(os.path.join(OUTPUT_DIR, "all_reaction_ids.txt"), "w") as f:
        f.write("\n".join(rxn_ids))
    return rxn_ids

# Step 2: Download each reaction entry
def download_reaction_entries(rxn_ids, delay=1.0):
    for rxn in rxn_ids:
        out_path = os.path.join(OUTPUT_DIR, f"reaction_{rxn}.txt")
        if os.path.exists(out_path):
            continue  # Skip if already downloaded
        url = f"https://rest.kegg.jp/get/rn:{rxn}"
        print(f"Fetching {url}...")
        r = requests.get(url)
        if r.status_code == 200:
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(r.text)
        else:
            print(f"❌ Failed to fetch {rxn}")
        time.sleep(delay)

# Run the full workflow
if __name__ == "__main__":
    rxn_ids = download_all_reaction_ids()
    download_reaction_entries(rxn_ids, delay=1.5)  # go slow to be polite


In [ ]:
### fetching the data



# --- CONFIGURATION ---
OUTPUT_DIR = '/Users/user/Documents/research/AAAIM/data/kegg/'
SLEEP_BETWEEN_REQUESTS = 2  # seconds
MAX_ENTRIES = 100  # change as needed

# --- SETUP ---
os.makedirs(OUTPUT_DIR, exist_ok=True)
BASE_URL = "https://rest.kegg.jp"

# --- HELPERS ---
def save_file(filename, content):
    with open(os.path.join(OUTPUT_DIR, filename), "w", encoding="utf-8") as f:
        f.write(content)

def get_and_save(endpoint, filename):
    print(f"Fetching {endpoint}...")
    r = requests.get(f"{BASE_URL}{endpoint}")
    if r.status_code == 200:
        save_file(filename, r.text)
        return r.text
    else:
        print(f"Failed to get {endpoint}: {r.status_code}")
        return None

# --- STEP 1: Get list of EC numbers ---
enzymes_txt = get_and_save("/list/enzyme", "enzyme_list.txt")
if not enzymes_txt:
    exit()

ec_numbers = [line.split()[0].replace("ec:", "") for line in enzymes_txt.strip().split("\n")][MAX_ENTRIES:]

# --- STEP 2: Download data for each EC ---
for ec in ec_numbers:
    print(f"\nProcessing EC:{ec}")
    
    # Download enzyme entry
    get_and_save(f"/get/ec:{ec}", f"enzyme_ec_{ec}.txt")
    
    # Download linked reactions
    get_and_save(f"/link/reaction/ec:{ec}", f"reaction_links_ec_{ec}.txt")

    # Download linked pathways
    get_and_save(f"/link/pathway/ec:{ec}", f"pathway_links_ec_{ec}.txt")
    
    time.sleep(SLEEP_BETWEEN_REQUESTS)

print("\n✅ KEGG data download complete.")


In [ ]:
def load_chebi_to_kegg_mapping(filepath):
    """Returns a dict mapping ChEBI ID → KEGG compound ID (Cxxxxx)"""
    chebi_to_kegg = {}
    with open(filepath, "r") as f:
        for line in f:
            if "\t" not in line:
                continue
            chebi, kegg = line.strip().split("\t")
            if chebi.startswith("chebi:") and kegg.startswith("cpd:"):
                chebi_id = chebi.replace("chebi:", "CHEBI:")
                kegg_id = kegg.replace("cpd:", "")
                chebi_to_kegg[chebi_id] = kegg_id
    return chebi_to_kegg

chebi_to_kegg_map = load_chebi_to_kegg_mapping('../kegg_data/chebi_kegg_mappings.txt')
with lzma.open("chebi_to_kegg_compound.lzma", "wt") as f:
    json.dump(chebi_to_kegg_map, f)

In [ ]:
INPUT_DIR = "../../kegg/reaction_flats"
OUTPUT_FILE = "./parsed_kegg_reactions.json"
parsed_reactions = []

def parse_kegg_flat_file(text):
    entry = {}
    current_key = None
    for line in text.splitlines():
        if line[:12].strip():  # new field
            current_key = line[:12].strip()
            entry[current_key] = line[12:].strip()
        elif current_key:
            entry[current_key] += " " + line[12:].strip()
    return entry

def extract_direction_and_compounds(equation):
    """Parses the EQUATION field to extract substrates, products, and direction."""
    if "<=>" in equation:
        direction = "reversible"
        lhs, rhs = equation.split("<=>")
    elif "=" in equation:
        direction = "irreversible"
        lhs, rhs = equation.split("=")
    else:
        return None, [], []

    def parse_side(side):
        compounds = []
        for item in side.strip().split("+"):
            match = re.search(r"(C\d{5})", item)
            if match:
                compounds.append(match.group(1))
        return compounds

    substrates = parse_side(lhs)
    products = parse_side(rhs)
    return direction, substrates, products

# Process each KEGG reaction file
for filepath in glob.glob(os.path.join(INPUT_DIR, "reaction_R*.txt")):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    raw = parse_kegg_flat_file(content)

    reaction_id = os.path.basename(filepath).replace("reaction_", "").replace(".txt", "")

    direction, substrates, products = extract_direction_and_compounds(raw.get("EQUATION", ""))

    record = {
        "reaction_id": f"{reaction_id}",
        "name": raw.get("NAME", ""),
        "ec_numbers": raw.get("ENZYME", "").split(),
        "direction": direction,
        "substrates": substrates,
        "products": products,
        "pathways": [p.strip().split()[0] for p in raw.get("PATHWAY", "").split("map") if p.strip()] if "PATHWAY" in raw else [],
        "raw_equation": raw.get("EQUATION", ""),
    }

    parsed_reactions.append(record)

# Write to JSON
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(parsed_reactions, f, indent=2)

print(f"✅ Parsed {len(parsed_reactions)} reactions → {OUTPUT_FILE}")


✅ Parsed 12304 reactions → ./parsed_kegg_reactions.json


In [21]:
def build_chunks_for_embedding(kegg_reactions):
    """Convert parsed KEGG reactions into text + metadata chunks for Chroma."""
    chunks = []

    for rxn in kegg_reactions:
        ec_line = f"EC: {', '.join(rxn['ec_numbers'])}" if rxn['ec_numbers'] else ""
        subs = ', '.join(rxn['substrates'])
        prods = ', '.join(rxn['products'])

        text = f"""KEGG Reaction {rxn['reaction_id']}
{ec_line}
Equation: {rxn['raw_equation']}
Substrates: {subs}
Products: {prods}
Pathways: {', '.join(rxn.get('pathways', []))}"""

        chunks.append({
            "reaction_id": rxn["reaction_id"],
            "text": text,
            "metadata": {
                "reaction_id": rxn["reaction_id"],
                "ec_numbers": ', '.join(rxn["ec_numbers"]),
                "substrates": ', '.join(rxn["substrates"]),
                "products": ', '.join(rxn["products"]),
                "pathways": ', '.join(rxn.get("pathways", []))
            }
        })

    return chunks


In [22]:
import chromadb
from sentence_transformers import SentenceTransformer

# Load and chunk parsed reactions
import json
with open("parsed_kegg_reactions.json", "r") as f:
    kegg_reactions = json.load(f)

chunks = build_chunks_for_embedding(kegg_reactions)
chunks[0]

{'reaction_id': 'R00001',
 'text': 'KEGG Reaction R00001\nEC: 3.6.1.10\nEquation: C00404 + n C00001 <=> (n+1) C02174\nSubstrates: C00404, C00001\nProducts: C02174\nPathways: ',
 'metadata': {'reaction_id': 'R00001',
  'ec_numbers': '3.6.1.10',
  'substrates': 'C00404, C00001',
  'products': 'C02174',
  'pathways': ''}}

In [ ]:
# Initialize Chroma persistent client
model = SentenceTransformer('all-MiniLM-L6-v2')  # or any other encoder
client = chromadb.PersistentClient(path="chroma_kegg_db")
collection = client.get_or_create_collection(name="kegg_reactions")


# Prepare data
ids = [chunk["reaction_id"] for chunk in chunks]
docs = [chunk["text"] for chunk in chunks]
metas = [chunk["metadata"] for chunk in chunks]

# Embed the documents
print("Embedding chunks...")
embeddings = model.encode(docs, show_progress_bar=True)

# Add to Chroma collection
print("Storing in Chroma...")
collection.add(
    ids=ids,
    documents=docs,
    embeddings=embeddings,
    metadatas=metas
)

print(f"✅ Stored {len(ids)} KEGG reaction chunks in Chroma.")


In [ ]:
def build_chunks_for_embedding(kegg_reactions):
    """Converts parsed KEGG JSON entries into semantically meaningful text chunks for vector embedding."""
    chunks = []

    for rxn in kegg_reactions:
        ec_line = f"EC: {', '.join(rxn['ec_numbers'])}" if rxn['ec_numbers'] else ""
        subs = ', '.join(rxn['substrates'])
        prods = ', '.join(rxn['products'])
        text = f"""KEGG Reaction {rxn['reaction_id']}
{ec_line}
Equation: {rxn['raw_equation']}
Substrates: {subs}
Products: {prods}
Pathways: {', '.join(rxn.get('pathways', []))}"""

        chunks.append({
            "reaction_id": rxn["reaction_id"],
            "text": text
        })

    return chunks

# embed using any encoder (e.g., OpenAI, HuggingFace):
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

In [24]:
from tqdm import tqdm

def batch_add_to_chroma(collection, chunks, model, batch_size=5000):
    ids = [c["reaction_id"] for c in chunks]
    docs = [c["text"] for c in chunks]
    metas = [c["metadata"] for c in chunks]

    print("🔍 Pre-embedding all reaction documents...")
    embeddings = model.encode(docs, show_progress_bar=True)

    print("🧠 Inserting into Chroma in batches...")
    for i in tqdm(range(0, len(ids), batch_size), desc="Storing in Chroma"):
        collection.add(
            ids=ids[i:i+batch_size],
            documents=docs[i:i+batch_size],
            metadatas=metas[i:i+batch_size],
            embeddings=embeddings[i:i+batch_size]
        )

    print(f"✅ Stored {len(ids)} KEGG reaction chunks in Chroma.")


In [25]:
model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="chroma_kegg_db")
collection = client.get_or_create_collection(name="kegg_reactions")

batch_add_to_chroma(collection, chunks, model)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


🔍 Pre-embedding all reaction documents...


Batches: 100%|██████████| 385/385 [00:36<00:00, 10.49it/s]


🧠 Inserting into Chroma in batches...


Storing in Chroma: 100%|██████████| 3/3 [00:04<00:00,  1.41s/it]

✅ Stored 12304 KEGG reaction chunks in Chroma.


In [3]:
def load_chebi_to_kegg_mapping(filepath):
    """Returns a dict mapping ChEBI ID → KEGG compound ID (Cxxxxx)"""
    chebi_to_kegg = {}
    with open(filepath, "r") as f:
        for line in f:
            if "\t" not in line:
                continue
            chebi, kegg = line.strip().split("\t")
            if chebi.startswith("chebi:") and kegg.startswith("cpd:"):
                chebi_id = chebi.replace("chebi:", "CHEBI:")
                kegg_id = kegg.replace("cpd:", "")
                chebi_to_kegg[chebi_id] = kegg_id
    return chebi_to_kegg
# Load the ChEBI → KEGG map
chebi_to_kegg = load_chebi_to_kegg_mapping("chebi_kegg_mappings.txt")


In [4]:
import json
import lzma

with lzma.open("chebi_to_kegg.json.lzma", "wt", encoding="utf-8") as f:
    json.dump(chebi_to_kegg, f, indent=2)

In [ ]:
# from load_data.py

def create_embeddings(
    ids: List[str],
    documents: List[str], 
    metadatas: List[Dict[str, Any]],
    collection_name: str,
    model_type: str = "default",
    persist_directory: str = "chroma_storage",
    batch_size: int = 500
) -> None:
    """
    Create embeddings and index documents using ChromaDB.
    
    Args:
        ids: List of document IDs
        documents: List of document texts
        metadatas: List of document metadata
        collection_name: Name for the ChromaDB collection
        model_type: Type of embedding model ("default", "openai")
        persist_directory: Directory to store the ChromaDB database
        batch_size: Number of documents to process in each batch
    """
    logger.info(f"Creating embeddings with {model_type} model...")
    
    # Initialize ChromaDB client
    client = chromadb.PersistentClient(path=persist_directory)
    
    # Get embedding function
    embedding_function = get_embedding_function(model_type)
    
    # Create or get collection
    collection = client.get_or_create_collection(
        name=collection_name,
        embedding_function=embedding_function,
        metadata={"model": model_type, "purpose": "entity_linking"}
    )
    
    # Index documents in batches
    total_docs = len(documents)
    logger.info(f"Indexing {total_docs} documents in batches of {batch_size}")
    
    try:
        for i, (id_batch, doc_batch, meta_batch) in enumerate(
            zip(
                chunk_list(ids, batch_size), 
                chunk_list(documents, batch_size), 
                chunk_list(metadatas, batch_size)
            )
        ):
            start_doc = i * batch_size
            end_doc = min((i + 1) * batch_size, total_docs)
            percent_done = (end_doc / total_docs) * 100 if total_docs > 0 else 0
            logger.info(
                f"Processing batch {i+1}, documents {start_doc} to {end_doc} "
                f"({percent_done:.2f}% complete)"
            )
            
            collection.add(
                ids=id_batch,
                documents=doc_batch,
                metadatas=meta_batch
            )
        
        final_count = collection.count()
        logger.info(f"Successfully indexed {final_count} documents in collection '{collection_name}'")
        logger.info(f"Collection saved to {persist_directory}")
        
    except Exception as e:
        logger.error(f"Error creating embeddings: {e}")
        raise

In [ ]:
print(collection.count())  # How many reactions are indexed?


In [4]:
import json
import lzma

with open('parsed_kegg_reactions.json', 'r', encoding='utf-8') as infile:
    data = json.load(infile)

with lzma.open('parsed_kegg_reactions.lzma', "wt", encoding="utf-8") as f:
    json.dump(data, f, indent=2)


In [ ]:
# download data from KEGG
import os
import requests
import time

# --- CONFIGURATION ---
OUTPUT_DIR = "kegg_data"
SLEEP_BETWEEN_REQUESTS = 2  # seconds
MAX_ENTRIES = 100  # change as needed

# --- SETUP ---
os.makedirs(OUTPUT_DIR, exist_ok=True)
BASE_URL = "https://rest.kegg.jp"

# --- HELPERS ---
def save_file(filename, content):
    with open(os.path.join(OUTPUT_DIR, filename), "w", encoding="utf-8") as f:
        f.write(content)

def get_and_save(endpoint, filename):
    print(f"Fetching {endpoint}...")
    r = requests.get(f"{BASE_URL}{endpoint}")
    if r.status_code == 200:
        save_file(filename, r.text)
        return r.text
    else:
        print(f"Failed to get {endpoint}: {r.status_code}")
        return None

# --- STEP 1: Get list of EC numbers ---
enzymes_txt = get_and_save("/list/enzyme", "enzyme_list.txt")
if not enzymes_txt:
    exit()

ec_numbers = [line.split()[0].replace("ec:", "") for line in enzymes_txt.strip().split("\n")][MAX_ENTRIES:]

# --- STEP 2: Download data for each EC ---
for ec in ec_numbers:
    print(f"\nProcessing EC:{ec}")
    
    # Download enzyme entry
    get_and_save(f"/get/ec:{ec}", f"enzyme_ec_{ec}.txt")
    
    # Download linked reactions
    get_and_save(f"/link/reaction/ec:{ec}", f"reaction_links_ec_{ec}.txt")

    # Download linked pathways
    get_and_save(f"/link/pathway/ec:{ec}", f"pathway_links_ec_{ec}.txt")
    
    time.sleep(SLEEP_BETWEEN_REQUESTS)

print("\n✅ KEGG data download complete.")


In [ ]:
# parse reactions into json file

import os
import json
import glob
import re

INPUT_DIR = "kegg_data/reactions_all"   # or wherever your files are
OUTPUT_FILE = "parsed_kegg_reactions.json"
parsed_reactions = []

def parse_kegg_flat_file(text):
    entry = {}
    current_key = None
    for line in text.splitlines():
        if line[:12].strip():  # new field
            current_key = line[:12].strip()
            entry[current_key] = line[12:].strip()
        elif current_key:
            entry[current_key] += " " + line[12:].strip()
    return entry

def extract_direction_and_compounds(equation):
    """Parses the EQUATION field to extract substrates, products, and direction."""
    if "<=>" in equation:
        direction = "reversible"
        lhs, rhs = equation.split("<=>")
    elif "=" in equation:
        direction = "irreversible"
        lhs, rhs = equation.split("=")
    else:
        return None, [], []

    def parse_side(side):
        compounds = []
        for item in side.strip().split("+"):
            match = re.search(r"(C\d{5})", item)
            if match:
                compounds.append(match.group(1))
        return compounds

    substrates = parse_side(lhs)
    products = parse_side(rhs)
    return direction, substrates, products

# Process each KEGG reaction file
for filepath in glob.glob(os.path.join(INPUT_DIR, "reaction_R*.txt")):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    raw = parse_kegg_flat_file(content)

    reaction_id = os.path.basename(filepath).replace("reaction_", "").replace(".txt", "")

    direction, substrates, products = extract_direction_and_compounds(raw.get("EQUATION", ""))

    record = {
        "reaction_id": f"R{reaction_id}",
        "name": raw.get("NAME", ""),
        "ec_numbers": raw.get("ENZYME", "").split(),
        "direction": direction,
        "substrates": substrates,
        "products": products,
        "pathways": [p.strip().split()[0] for p in raw.get("PATHWAY", "").split("map") if p.strip()] if "PATHWAY" in raw else [],
        "raw_equation": raw.get("EQUATION", ""),
    }

    parsed_reactions.append(record)

# Write to JSON
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(parsed_reactions, f, indent=2)

print(f"✅ Parsed {len(parsed_reactions)} reactions → {OUTPUT_FILE}")
